In [1]:
# ── 1. Install ────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pickle
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [3]:
# ── 3. Load RGB predictions ───────────────────────────────────────────────────
RGB_PROBS_PATH  = '/content/drive/MyDrive/violence-detection/checkpoints/rgb_v2/rgb_v2_val_probs.npy'
RGB_LABELS_PATH = '/content/drive/MyDrive/violence-detection/checkpoints/rgb_v2/rgb_v2_val_labels.npy'

rgb_probs_arr  = np.load(RGB_PROBS_PATH)
rgb_labels_arr = np.load(RGB_LABELS_PATH)

print("RGB probs shape:  ", rgb_probs_arr.shape)
print("RGB only accuracy:", accuracy_score(rgb_labels_arr,
                                           np.argmax(rgb_probs_arr, axis=1)))

RGB probs shape:   (400, 2)
RGB only accuracy: 0.88


In [4]:
# ── 4. Load Joint predictions (already saved from 3-stream notebook) ──────────
JOINT_PATH = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-joint/joint_val_predictions.pkl'

with open(JOINT_PATH, 'rb') as f:
    joint_preds = pickle.load(f)

joint_probs_arr  = np.array([p['pred_score'].numpy() for p in joint_preds])
joint_labels_arr = np.array([p['gt_label'].item()    for p in joint_preds])

print("Joint probs shape: ", joint_probs_arr.shape)
print("Joint only accuracy:", accuracy_score(joint_labels_arr,
                                             np.argmax(joint_probs_arr, axis=1)))

Joint probs shape:  (400, 2)
Joint only accuracy: 0.8075


In [5]:
# ── 5. Check alignment ────────────────────────────────────────────────────────
print("Labels match:", np.all(rgb_labels_arr == joint_labels_arr))
assert np.all(rgb_labels_arr == joint_labels_arr), "Label mismatch!"
labels = joint_labels_arr
print("Streams aligned ✅")

Labels match: True
Streams aligned ✅


In [6]:
# ── 6. RGB + Joint grid search fusion ────────────────────────────────────────
print("=" * 55)
print(f"{'Weights (RGB / Joint)':<30} {'Acc':>10} {'F1':>10}")
print("=" * 55)

best_acc_rj   = -1
best_f1_rj    = -1
best_w_rgb    = -1
best_preds_rj = None

for w_rgb in np.arange(0.0, 1.05, 0.05):
    w_rgb   = round(float(w_rgb), 2)
    w_joint = round(1.0 - w_rgb, 2)
    fused   = w_rgb * rgb_probs_arr + w_joint * joint_probs_arr
    preds   = np.argmax(fused, axis=1)
    acc     = accuracy_score(labels, preds)
    f1      = f1_score(labels, preds)
    print(f"RGB {w_rgb:.2f} + Joint {w_joint:.2f}         {acc:.4f}     {f1:.4f}")
    if acc > best_acc_rj or (acc == best_acc_rj and f1 > best_f1_rj):
        best_acc_rj   = acc
        best_f1_rj    = f1
        best_w_rgb    = w_rgb
        best_preds_rj = preds

print("=" * 55)
print(f"Best: RGB {best_w_rgb:.2f} + Joint {round(1-best_w_rgb,2):.2f}")
print(f"Best accuracy : {best_acc_rj:.4f}  →  {best_acc_rj*100:.2f}%")
print(f"Best F1       : {best_f1_rj:.4f}")

Weights (RGB / Joint)                 Acc         F1
RGB 0.00 + Joint 1.00         0.8075     0.8089
RGB 0.05 + Joint 0.95         0.8050     0.8060
RGB 0.10 + Joint 0.90         0.8100     0.8109
RGB 0.15 + Joint 0.85         0.8175     0.8170
RGB 0.20 + Joint 0.80         0.8175     0.8161
RGB 0.25 + Joint 0.75         0.8225     0.8229
RGB 0.30 + Joint 0.70         0.8300     0.8308
RGB 0.35 + Joint 0.65         0.8350     0.8358
RGB 0.40 + Joint 0.60         0.8425     0.8437
RGB 0.45 + Joint 0.55         0.8425     0.8444
RGB 0.50 + Joint 0.50         0.8550     0.8564
RGB 0.55 + Joint 0.45         0.8600     0.8614
RGB 0.60 + Joint 0.40         0.8525     0.8536
RGB 0.65 + Joint 0.35         0.8600     0.8621
RGB 0.70 + Joint 0.30         0.8775     0.8790
RGB 0.75 + Joint 0.25         0.8800     0.8818
RGB 0.80 + Joint 0.20         0.8825     0.8840
RGB 0.85 + Joint 0.15         0.8825     0.8845
RGB 0.90 + Joint 0.10         0.8800     0.8829
RGB 0.95 + Joint 0.05         0.880

In [7]:
# ── 7. Detailed report for best fusion ───────────────────────────────────────
print(f"\n=== RGB {best_w_rgb:.2f} + Joint {round(1-best_w_rgb,2):.2f} (Best) ===")
print(classification_report(labels, best_preds_rj,
                             target_names=['NonFight', 'Fight']))


=== RGB 0.85 + Joint 0.15 (Best) ===
              precision    recall  f1-score   support

    NonFight       0.90      0.86      0.88       200
       Fight       0.87      0.90      0.88       200

    accuracy                           0.88       400
   macro avg       0.88      0.88      0.88       400
weighted avg       0.88      0.88      0.88       400

